# Routing — OSRM via HAProxy

Demonstrates the `route()` function from the `airgap_geo` library, which calls
**OSRM** through an HAProxy reverse proxy to compute driving, walking, and cycling
routes between two points.

For enhanced routing features (waypoints, turn-by-turn steps, alternative routes,
annotations, road-class exclusions), see
[03-routing-enhanced.ipynb](03-routing-enhanced.ipynb).

## Prerequisites

| Service        | Default port | Role                        |
| -------------- | ------------ | --------------------------- |
| OSRM + HAProxy | 80           | Road routing (all profiles) |
| Nominatim      | 8080         | Used to geocode origin/dest |
| Photon         | 2322         | Enriches geocoding results  |

Start the services manually (see `docker/README.md`), or run the **Start Services**
cell below.


______________________________________________________________________

## Start Services (optional)

Run the cell below to start the combined stack and wait until routing and geocoding
services respond. **Skip if the services are already running.**


In [ ]:
import pathlib
import subprocess
import time

import requests as _requests

from airgap_geo.settings import NOMINATIM_URL, OSRM_API, PHOTON_API


def _find_repo_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


_REPO_ROOT = _find_repo_root(pathlib.Path().resolve())
_COMPOSE_FILE = _REPO_ROOT / "docker" / "docker-compose.yml"
_ENV_FILE = _REPO_ROOT / ".env"

_HEALTH_TIMEOUT = 120

_ENDPOINTS = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
    "OSRM / HAProxy": OSRM_API,
}


def start_services(timeout: int = _HEALTH_TIMEOUT) -> None:  # noqa: D103
    print(f"Starting stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "up",
            "-d",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    ready = {name: False for name in _ENDPOINTS}
    deadline = time.monotonic() + timeout
    print(f"\nPolling services (timeout {timeout}s) ...")

    while time.monotonic() < deadline:
        for name, url in _ENDPOINTS.items():
            if ready[name]:
                continue
            try:
                _requests.get(url, timeout=3)
                ready[name] = True
                print(f"  \u2713  {name} is up  ({url})")
            except Exception:
                pass
        if all(ready.values()):
            break
        time.sleep(3)

    still_down = [n for n, ok in ready.items() if not ok]
    if still_down:
        print(f"\n[WARNING] Timed out waiting for: {', '.join(still_down)}")
    else:
        print("\nAll services are up and ready.")


start_services()

## Setup — Imports and Configuration


In [ ]:
import importlib

import folium
import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder, route  # noqa: E402
from airgap_geo.settings import OSRM_API  # noqa: E402

client = httpx.AsyncClient()

print(f"OSRM / HAProxy : {OSRM_API}")

## Health Check


In [ ]:
try:
    r = requests.get(OSRM_API, timeout=5)
    print(f"OSRM / HAProxy  \u2713  HTTP {r.status_code}  ({OSRM_API})")
except Exception:
    print(f"OSRM / HAProxy  \u2717  unreachable  ({OSRM_API})")

______________________________________________________________________

## 1 — Geocode Origin and Destination


In [ ]:
result_origin = await geocoder("King's Cross Station, London", client)
result_dest = await geocoder("Victoria Station, London", client)

origin_coords = (result_origin["geo"]["lat"], result_origin["geo"]["lon"])
dest_coords = (result_dest["geo"]["lat"], result_dest["geo"]["lon"])

print(f"Origin      : King's Cross  \u2192 {origin_coords}")
print(f"Destination : Victoria      \u2192 {dest_coords}")

______________________________________________________________________

## 2 — Compare All Three Profiles

| Profile   | Backend container | Typical use              |
| --------- | ----------------- | ------------------------ |
| `driving` | `osrm-driving`    | Car routes, fastest path |
| `walking` | `osrm-foot`       | Pedestrian routes        |
| `cycling` | `osrm-bike`       | Bicycle routes           |


In [ ]:
_PROFILES = ["driving", "walking", "cycling"]
route_results = {}
rows = []

for profile in _PROFILES:
    r = await route(origin_coords, dest_coords, client, profile=profile)
    route_results[profile] = r
    if r and r.get("routes"):
        leg = r["routes"][0]
        rows.append(
            {
                "Profile": profile.capitalize(),
                "Distance (km)": round(leg["distance"] / 1000, 2),
                "Duration (min)": round(leg["duration"] / 60, 1),
            }
        )
    else:
        rows.append(
            {
                "Profile": profile.capitalize(),
                "Distance (km)": "N/A",
                "Duration (min)": "N/A",
            }
        )

pd.DataFrame(rows).set_index("Profile")

In [ ]:
# Demonstrate that an unsupported profile raises ValueError
try:
    await route(origin_coords, dest_coords, client, profile="flying")
except ValueError as exc:
    print(f"ValueError raised as expected:\n  {exc}")

______________________________________________________________________

## 3 — Map the Driving Route


In [ ]:
def _decode_polyline(encoded: str) -> list[tuple[float, float]]:
    """Decode an OSRM / Google encoded polyline string into (lat, lon) pairs."""
    coords: list[tuple[float, float]] = []
    index = 0
    lat = 0
    lng = 0
    while index < len(encoded):
        for is_lng in (False, True):
            shift, result = 0, 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1F) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = ~(result >> 1) if (result & 1) else (result >> 1)
            if is_lng:
                lng += delta
            else:
                lat += delta
        coords.append((lat / 1e5, lng / 1e5))
    return coords


driving_data = route_results.get("driving", {})
centre_lat = (origin_coords[0] + dest_coords[0]) / 2
centre_lon = (origin_coords[1] + dest_coords[1]) / 2

m_route = folium.Map(location=[centre_lat, centre_lon], zoom_start=13)

folium.Marker(
    list(origin_coords),
    tooltip="King's Cross (origin)",
    icon=folium.Icon(color="green", icon="play"),
).add_to(m_route)
folium.Marker(
    list(dest_coords),
    tooltip="Victoria (destination)",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m_route)

if driving_data and driving_data.get("routes"):
    encoded = driving_data["routes"][0].get("geometry", "")
    if encoded:
        decoded = _decode_polyline(encoded)
        folium.PolyLine(
            decoded, color="royalblue", weight=4, tooltip="Driving route"
        ).add_to(m_route)

m_route

______________________________________________________________________

## 4 — Error Handling

- **Invalid routing profile** \\u2192 `ValueError`
- **OSRM unreachable / non-200** \\u2192 returns `None`


In [ ]:
# Invalid profile raises ValueError
try:
    await route((51.5, -0.1), (52.5, -1.9), client, profile="teleportation")
except ValueError as exc:
    print(f"Bad profile  \u2192 ValueError: {exc}")

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:  # noqa: D103
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()